# Step 5: Ablation study

Each variant removes or isolates one group of candidate features and is **retrained from scratch**. The user–article matching group U stays in every variant, so changes come from candidate-side evidence only.

| Variant | T1 behaviour | T2 transferred | T3 content | U matching | H content_smoothed_ctr |
|---|:-:|:-:|:-:|:-:|:-:|
| `full` | ✓ | ✓ | ✓ | ✓ | |
| `no_t1` | | ✓ | ✓ | ✓ | |
| `no_t2` | ✓ | | ✓ | ✓ | |
| `no_t3` | ✓ | ✓ | | ✓ | |
| `t1_only` | ✓ | | | ✓ | |
| `t3_only` | | | ✓ | ✓ | |
| `full_plus_smoothing` | ✓ | ✓ | ✓ | ✓ | ✓ |

**Protocol.** Every variant uses the tree shape selected in step 4 and its own early stopping on validation. For `full_plus_smoothing`, α is chosen on validation from {5, 10, 20, 50, 100} with the shared selection rule. Test is scored once per variant, after all choices are fixed.

"Without Tier 3" removes the explicit content columns only. Content still reaches the model through Tier 2 (transferred behaviour) and U (similarities), so it measures the value of the explicit content inputs given everything else. Contributions need not add up, because groups overlap.

> Metrics here come from `shared/eval_interim.py`. Final ablation tables, segment analysis and uncertainty come from `step6_evaluation_diagnostics.ipynb`.

## 1. Setup

In [ ]:
import os, sys, subprocess, json, time

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Bryng23/BT4222-Group-11.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/BT4222-Group-11"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    DATA_ROOT = "/content/drive/MyDrive/BT4222_G11"
else:
    REPO_DIR = os.getcwd()
    DATA_ROOT = os.path.join(REPO_DIR, "data")
sys.path.insert(0, REPO_DIR)

import numpy as np
import pandas as pd
from shared import schema as S, feature_groups as FG, lgbm_pipeline as L, selection as SEL
from shared.predictions import write_predictions
from shared import eval_interim as E      # swap for the team evaluator once it exists

DATA_KIND = os.environ.get("G11_DATA_KIND", "REAL")   # SYNTHETIC only for local development
if DATA_KIND == "REAL":
    import kagglehub
    RAW_DIR = os.path.join(kagglehub.dataset_download("arashnic/mind-news-dataset"), "MINDsmall_train")
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"v{S.SCHEMA_VERSION}")
else:
    FEATURE_ROOT = os.path.join(DATA_ROOT, "features", f"synthetic_v{S.SCHEMA_VERSION}", "learnable")
    RAW_DIR = os.path.join(FEATURE_ROOT, "raw")
version = S.read_version(FEATURE_ROOT)
assert version.get("data_kind", "REAL") == DATA_KIND, f"feature tables are {version.get('data_kind')}, run is {DATA_KIND}"
PRED_ROOT = os.path.join(DATA_ROOT, "predictions")
MODEL_DIR = os.path.join(DATA_ROOT, "models", DATA_KIND.lower())
os.makedirs(MODEL_DIR, exist_ok=True)
SEED = 4222
N_TITLE_PC, N_ENT_PC = 32, 16
INCLUDE_USER_PC = True      # user profile vectors projected with the article PCA; set False if RAM runs short
SHOW = ["auc", "mrr", "ndcg@5", "ndcg@10", "cold_recall@10", "cold_hit@10", "warm_recall@10"]
print(f"DATA_KIND={DATA_KIND} | features: {FEATURE_ROOT} | schema {version['schema_version']}")

with open(os.path.join(MODEL_DIR, "lgbm_selected_params.json")) as f:
    chosen = json.load(f)
assert (chosen["n_title_pc"], chosen["n_ent_pc"], chosen["include_user_pc"]) == (N_TITLE_PC, N_ENT_PC, INCLUDE_USER_PC)
PARAMS = chosen["params"]
print("tree shape from step 4:", PARAMS)

## 2. Reduce vectors and load the row tables (same as step 4)

In [ ]:
t0 = time.time()
train_meta_ids = pd.read_parquet(S.meta_path(FEATURE_ROOT, "train"), columns=["impression_id", "news_id"])
accessible = set(train_meta_ids["news_id"].astype(str)) | L.train_histories(RAW_DIR, train_meta_ids["impression_id"].unique())
reducer = L.VectorReducer(N_TITLE_PC, N_ENT_PC, seed=SEED).fit(FEATURE_ROOT, accessible)
print(f"PCA fitted on {reducer.n_fit_articles:,} training-accessible articles "
      f"({reducer.n_fit_entity_articles:,} with entity vectors); explained variance {reducer.explained_variance()}")

data = {split: L.load_split(FEATURE_ROOT, split, reducer, include_user_pc=INCLUDE_USER_PC) for split in S.SPLITS}
INCLUDE_SEMANTIC = "semantic_neighbor_ctr" in data["train"][1].columns   # optional in the contract
GROUPS = FG.group_columns(N_TITLE_PC, N_ENT_PC, include_user_pc=INCLUDE_USER_PC, include_semantic=INCLUDE_SEMANTIC)
print("semantic_neighbor_ctr present:", INCLUDE_SEMANTIC)
pd.DataFrame({split: {"rows": len(m), "impressions": m["impression_id"].nunique(), "columns available": x.shape[1],
                      "memory MB": round(x.memory_usage(deep=True).sum() / 1e6, 1)}
              for split, (m, x) in data.items()}).T

## 3. Choose α for `full_plus_smoothing` on validation

In [ ]:
(m_tr, x_tr), (m_va, x_va) = data["train"], data["val"]
cols_h = FG.feature_columns("full_plus_smoothing", n_title_pc=N_TITLE_PC, n_ent_pc=N_ENT_PC, include_user_pc=INCLUDE_USER_PC,
                         include_semantic=INCLUDE_SEMANTIC)
alpha_rows, alpha_models = [], {}
for a in S.ALPHA_GRID:
    b = L.fit(L.design(x_tr, cols_h, a), m_tr["label"], L.design(x_va, cols_h, a), m_va["label"], PARAMS, seed=SEED)
    alpha_models[a] = b
    res = E.evaluate(m_va, L.predict(b, L.design(x_va, cols_h, a)))
    alpha_rows.append({"alpha": a, "best_iteration": b.best_iteration, **{k: res[k] for k in SHOW}})
alpha_table = pd.DataFrame(alpha_rows).set_index("alpha")
ALPHA = int(SEL.select(alpha_table))
print(f"selected alpha = {ALPHA}: {SEL.describe()}")
alpha_table.round(4)

## 4. Train every variant and score all splits

In [ ]:
per_imp, rows = {}, []
for variant in FG.ABLATIONS:
    cols = FG.feature_columns(variant, n_title_pc=N_TITLE_PC, n_ent_pc=N_ENT_PC, include_user_pc=INCLUDE_USER_PC,
                         include_semantic=INCLUDE_SEMANTIC)
    a = ALPHA if "content_smoothed_ctr" in cols else None
    t0 = time.time()
    b = alpha_models[ALPHA] if variant == "full_plus_smoothing" else \
        L.fit(L.design(x_tr, cols, a), m_tr["label"], L.design(x_va, cols, a), m_va["label"], PARAMS, seed=SEED)
    cfg = L.param_id(PARAMS) + (f",alpha={a}" if a else "")
    for split, (m, x) in data.items():
        sc = L.predict(b, L.design(x, cols, a))
        write_predictions(m["row_id"], sc, root=PRED_ROOT, data_kind=DATA_KIND, model_id=f"lgbm_{variant}", config_id=cfg,
                          split=split, schema_version=S.SCHEMA_VERSION,
                          extra={"variant": variant, "n_columns": len(cols), "best_iteration": b.best_iteration, "selected_on": "val"})
        pi = E.per_impression_metrics(m, sc)
        per_imp[(variant, split)] = pi
        rows.append({"variant": variant, "split": split, "n_columns": len(cols), "best_iteration": b.best_iteration,
                     **{k: v for k, v in E.summarise(pi).items() if k in SHOW}})
    print(f"{variant:20s} {len(cols):3d} columns, {b.best_iteration} trees, {time.time() - t0:.0f}s")
ablation = pd.DataFrame(rows).set_index(["split", "variant"])
ablation.loc["val"].round(4)

In [ ]:
ablation.loc["test"].round(4)

## 5. Change against the full model

Each variant minus `full` on the same impressions. For Cold Recall@10 and nDCG@10 the 95% interval is a paired bootstrap over impressions (1,000 resamples). Read validation for decisions; test confirms them.

In [ ]:
delta_rows = []
for split in ("val", "test"):
    base = per_imp[("full", split)]
    for variant in FG.ABLATIONS:
        if variant == "full":
            continue
        other = per_imp[(variant, split)]
        row = {"split": split, "variant": variant}
        for metric in ("ndcg@10", "cold_recall@10"):
            bs = E.paired_bootstrap(base[metric], other[metric])
            row.update({f"Δ {metric}": bs["mean_diff"], f"{metric} CI": f"[{bs['ci_low']:+.4f}, {bs['ci_high']:+.4f}]"})
        row["cold-click impressions"] = int(base["cold_recall@10"].notna().sum())
        delta_rows.append(row)
pd.DataFrame(delta_rows).set_index(["split", "variant"]).round(4)

## 6. Reading the results

* A large drop in Cold Recall@10 for `no_t2` or `no_t3`, with an interval below zero, is evidence that transferred behaviour or explicit content helps under-exposed articles.
* `no_t1` shows how much the model depends on the candidate's own history; its warm metrics should fall most.
* `t1_only` against `t3_only` contrasts pure behaviour with pure content, both on top of the same user matching.
* `full_plus_smoothing` tests whether the hand-built content-smoothed CTR adds anything the trees could not already build from T1 and T2.